In [1]:
import os
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import s3fs
import warnings
from shapely.geometry import box
from datetime import datetime
import zarr
import dask
import rioxarray
from shapely.geometry import mapping

## Download AORC

Variables available:

|Variable Name|Common Name|Description|
|---|---|---|
| APCP_surface | Total Precipitation | Hourly total precipitation (kgm-2 or mm) |
| TMP_2maboveground | Air Temperature | Temperature (at 2 m above-ground-level (AGL)) (K) |
| SPFH_2maboveground | Specific Humidity | Specific humidity (at 2 m AGL) (g g-1) |
| DLWRF_surface | Downward Long-Wave Radiation Flux | longwave (infrared) radiation flux (at the surface) (W m-2) |
| DSWRF_surface | Downward Short-Wave Radiation Flux | Downward shortwave (solar) radiation flux (at the surface) (W m-2) |
| PRES_surface | Pressure| Air pressure (at the surface) (Pa) |
| UGRD_10maboveground | U-Component of Wind | (west to east) - components of the wind (at 10 m AGL) (m s-1) |
| VGRD_10maboveground | V-Component of Wind | (south to north) - components of the wind (at 10 m AGL) (m s-1)|

In [2]:
def convert_dict_to_df(dict_of_dfs):
    from functools import reduce
    r = [df.rename(columns={'value': k}) for k, df in dict_of_dfs.items()]
    common_cols = [c for c in r[0].columns if c != list(dict_of_dfs.keys())[0] and c != 'value']
    return reduce(lambda l, r_: pd.merge(l, r_, on=common_cols), r)

def get_all_aorc_vars(AOI, Date_range, aorc_variables, tower_location=None):

    warnings.filterwarnings("ignore", category=UserWarning)
    
    # Start date & End date  - In Year-Month-Day format the earliest start date can be '1979-02-01'
    start_datetime = Date_range[0]; end_datetime = Date_range[1]
    ## Create a list of years to retrieve data 
    start_yr = datetime.strptime(start_datetime, '%Y-%m-%d').year
    end_yr = datetime.strptime(end_datetime, '%Y-%m-%d').year
    yrs = list(range(start_yr, end_yr+1))

    ## Loading data (AORC data are organized by years, look at https://noaa-nws-aorc-v1-1-1km.s3.amazonaws.com/index.html)
    # Base URL
    base_url = f's3://noaa-nws-aorc-v1-1-1km'
    # Creating a connection to Amazon S3 bucket using the s3fs library (https://s3fs.readthedocs.io/en/latest/api.html).
    s3_out = s3fs.S3FileSystem(anon=True)              # access S3 as if it were a file system. 
    fileset = [s3fs.S3Map(                             # maps each year's Zarr dataset from S3 to a local-like object.
            root=f"s3://{base_url}/{yr}.zarr",     # Zarr dataset for each year
            s3=s3_out,                             # connection
            check=False                            # checking if the dataset exists before trying to load it
        ) for yr in yrs]                           # loops through each year

    ## Load data for specified years and veriable of interest using the xarray library
    ds_yrs = xr.open_mfdataset(fileset, engine='zarr')
    all_vars_dict = {}
    for variable_name in aorc_variables:
        da_yrs_var = ds_yrs[variable_name]
        da_yrs_var_subset = da_yrs_var.sel(time=slice(start_datetime, end_datetime))

        # clip to AOI
        AOI = AOI.to_crs(da_yrs_var_subset.rio.crs)  # Reproject if needed
        da_yrs_var_subset.rio.write_crs(AOI.crs, inplace=True)  # Set CRS on DataArray
        
        try:
            da_yrs_var_subset_clipped = da_yrs_var_subset.rio.clip(
                geometries=AOI.geometry.apply(mapping),
                crs=AOI.crs
            )
        except:
            # get the nearest point to the center of the field if no point in AOI
            print("AOI not overlapping grid — using nearest grid point instead.")
            da_yrs_var_subset_clipped = da_yrs_var_subset.sel(latitude=tower_location[0], longitude=tower_location[1], method='nearest')

        # Convert to df, rename the value col, save to dict
        all_vars_dict[variable_name] = da_yrs_var_subset_clipped.to_dataframe(name="value").reset_index()


    return convert_dict_to_df(all_vars_dict)



In [3]:
AORC_variables = ['APCP_surface', 'TMP_2maboveground', 'SPFH_2maboveground', 'DLWRF_surface',
                   'DSWRF_surface', 'PRES_surface', 'UGRD_10maboveground', 'VGRD_10maboveground']

variables_conversion = {
    'Total Precipitation': 'APCP_surface',
    'Air Temperature': 'TMP_2maboveground',
    'Specific Humidity': 'SPFH_2maboveground',
    'Downward Long-Wave Radiation Flux': 'DLWRF_surface',
    'Downward Short-Wave Radiation Flux': 'DSWRF_surface',
    'Pressure': 'PRES_surface',
    'U-Component of Wind': 'UGRD_10maboveground',
    'V-Component of Wind': 'VGRD_10maboveground'
}

#### Tower locations

In [48]:
# PA
HWB_2016_2018_AORC_df = get_all_aorc_vars(gpd.read_file('US-HWB_Field.geojson'), Date_range=('2016-01-01', '2018-01-01'),
                                           aorc_variables=AORC_variables,tower_location = (40.8608, -77.8488))\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})

HWB_2016_2018_AORC_df.to_csv('HWB_2016_2018_AORC.csv')


# CA
Bi1_2018_2024_AORC_df = get_all_aorc_vars(gpd.read_file('US-Bi1_Field.geojson'), Date_range=('2018-01-01', '2024-11-01'),
                                           aorc_variables=AORC_variables,tower_location = (38.0992, -121.4993))\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})
Bi1_2018_2024_AORC_df.to_csv('Bi1_2018_2024_AORC.csv')

Bi2_2018_2024_AORC_df = get_all_aorc_vars(gpd.read_file('US-Bi2_Field.geojson'), Date_range=('2018-01-01', '2024-11-01'),
                                           aorc_variables=AORC_variables,tower_location = (38.1091, -121.5351))\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})
Bi2_2018_2024_AORC_df.to_csv('Bi2_2018_2024_AORC.csv')

Tw3_2017_2018_AORC_df = get_all_aorc_vars(gpd.read_file('US-Tw3_Field.geojson'), Date_range=('2017-01-01', '2019-01-01'),
                                           aorc_variables=AORC_variables,tower_location = (38.1152, -121.6469))\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})
Tw3_2017_2018_AORC_df.to_csv('Tw3_2017_2018_AORC.csv')


# IL
UiABC_2017_2024_AORC_df = get_all_aorc_vars(gpd.read_file('US-UiABC_Fields.geojson'), Date_range=('2017-01-01', '2024-11-30'),
                                           aorc_variables=AORC_variables,tower_location = (40.0646, -88.1961))\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})
UiABC_2017_2024_AORC_df.to_csv('UiABC_2017_2024_AORC.csv')


# IN
VT12_2023_2024_AORC_df = get_all_aorc_vars(gpd.read_file('US-VT12_Fields.geojson'), Date_range=('2023-01-01', '2024-11-30'),
                                           aorc_variables=AORC_variables,tower_location = (40.4064, -87.5214))\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})

VT12_2023_2024_AORC_df.to_csv('UVT12_2023_2024_AORC.csv')

AOI not overlapping grid — using nearest grid point instead.
AOI not overlapping grid — using nearest grid point instead.
AOI not overlapping grid — using nearest grid point instead.
AOI not overlapping grid — using nearest grid point instead.
AOI not overlapping grid — using nearest grid point instead.
AOI not overlapping grid — using nearest grid point instead.
AOI not overlapping grid — using nearest grid point instead.
AOI not overlapping grid — using nearest grid point instead.


#### For select dates

In [4]:
AORC_variables = ['APCP_surface', 'TMP_2maboveground', 'SPFH_2maboveground', 'DLWRF_surface',
                   'DSWRF_surface', 'PRES_surface', 'UGRD_10maboveground', 'VGRD_10maboveground']

variables_conversion = {
    'Total Precipitation': 'APCP_surface',
    'Air Temperature': 'TMP_2maboveground',
    'Specific Humidity': 'SPFH_2maboveground',
    'Downward Long-Wave Radiation Flux': 'DLWRF_surface',
    'Downward Short-Wave Radiation Flux': 'DSWRF_surface',
    'Pressure': 'PRES_surface',
    'U-Component of Wind': 'UGRD_10maboveground',
    'V-Component of Wind': 'VGRD_10maboveground'
}

In [5]:
# PA
GBF_2023_AORC_df = get_all_aorc_vars(gpd.read_file(os.path.join(os.getcwd(), 'For_select_dates', 'Farm_extents', 'US-UC1_extent.geojson')),
                                                    Date_range=('2023-05-01', '2023-10-15'),
                                           aorc_variables=AORC_variables,tower_location = None)\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})

GBF_2023_AORC_df.to_csv('GBF_extent_2023_AORC.csv')


# CA
Bi1_2024_AORC_df = get_all_aorc_vars(gpd.read_file(os.path.join(os.getcwd(), 'For_select_dates', 'Farm_extents', 'US-Bi1_extent.geojson')),
                                                    Date_range=('2024-05-01', '2024-10-15'),
                                           aorc_variables=AORC_variables,tower_location = None)\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})

Bi1_2024_AORC_df.to_csv('Bi1_extent_2024_AORC.csv')

Bi2_2024_AORC_df = get_all_aorc_vars(gpd.read_file(os.path.join(os.getcwd(), 'For_select_dates', 'Farm_extents', 'US-Bi2_extent.geojson')),
                                                    Date_range=('2024-05-01', '2024-10-15'),
                                           aorc_variables=AORC_variables,tower_location = None)\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})

Bi2_2024_AORC_df.to_csv('Bi2_extent_2024_AORC.csv')

Tw3_2024_AORC_df = get_all_aorc_vars(gpd.read_file(os.path.join(os.getcwd(), 'For_select_dates', 'Farm_extents', 'US-Tw3_extent.geojson')),
                                                    Date_range=('2024-05-01', '2024-10-15'),
                                           aorc_variables=AORC_variables,tower_location = None)\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})

Tw3_2024_AORC_df.to_csv('Tw3_extent_2024_AORC.csv')


# IL
UiABC_2024_AORC_df = get_all_aorc_vars(gpd.read_file(os.path.join(os.getcwd(), 'For_select_dates', 'Farm_extents', 'US-UiABC_extent.geojson')),
                                                    Date_range=('2024-05-01', '2024-10-15'),
                                           aorc_variables=AORC_variables,tower_location = None)\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})

UiABC_2024_AORC_df.to_csv('UiABC_extent_2024_AORC.csv')


# IN
VT12_2024_AORC_df = get_all_aorc_vars(gpd.read_file(os.path.join(os.getcwd(), 'For_select_dates', 'Farm_extents', 'US-VT12_extent.geojson')),
                                                    Date_range=('2024-05-01', '2024-10-15'),
                                           aorc_variables=AORC_variables,tower_location = None)\
                                            .rename(columns={v: k for k, v in variables_conversion.items()})

VT12_2024_AORC_df.to_csv('VT12_extent_2024_AORC.csv')

## Previuos code

In [52]:
# Start date - In Year-Month-Day format the earliest start date can be '1979-02-01'
start_datetime = '2020-01-01'
# End date - In Year-Month-Day format the latest end date can be '2023-01-31'
end_datetime = '2024-12-31'

# File path to the shapefile
shapefile_path = "US-Bi1_Field.geojson"

# Variable name to retrieve data (look at the following table for valid variable names)
variable_name = 'APCP_surface'

# User-defined aggregation interval - valid values are 'hour','day','month','year'
agg_interval = 'hour'

In [53]:
# Read the shapefile
gdf = gpd.read_file(shapefile_path)

In [54]:
## Create a list of years to retrieve data 
start_yr = datetime.strptime(start_datetime, '%Y-%m-%d').year
end_yr = datetime.strptime(end_datetime, '%Y-%m-%d').year
yrs = list(range(start_yr, end_yr+1))

## Loading data (AORC data are organized by years, look at https://noaa-nws-aorc-v1-1-1km.s3.amazonaws.com/index.html)
# Base URL
base_url = f's3://noaa-nws-aorc-v1-1-1km'
# Creating a connection to Amazon S3 bucket using the s3fs library (https://s3fs.readthedocs.io/en/latest/api.html).
s3_out = s3fs.S3FileSystem(anon=True)              # access S3 as if it were a file system. 
fileset = [s3fs.S3Map(                             # maps each year's Zarr dataset from S3 to a local-like object.
            root=f"s3://{base_url}/{yr}.zarr",     # Zarr dataset for each year
            s3=s3_out,                             # connection
            check=False                            # checking if the dataset exists before trying to load it
        ) for yr in yrs]                           # loops through each year

## Load data for specified years and veriable of interest using the xarray library
ds_yrs = xr.open_mfdataset(fileset, engine='zarr')
da_yrs_var = ds_yrs[variable_name]
variable_long_name = da_yrs_var.attrs.get('long_name')
da_yrs_var

<xarray.DataArray 'APCP_surface' (time: 43848, latitude: 4201, longitude: 8401)> Size: 12TB
dask.array<concatenate, shape=(43848, 4201, 8401), dtype=float64, chunksize=(144, 128, 256), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 34kB 20.0 20.01 20.02 ... 54.98 54.99 55.0
  * longitude  (longitude) float64 67kB -130.0 -130.0 -130.0 ... -60.01 -60.0
  * time       (time) datetime64[ns] 351kB 2020-01-01 ... 2024-12-31T23:00:00
Attributes:
    AORC_Contact:  aorc.info@noaa.gov
    aorc_version:  v1.1
    crs:           EPSG:4326
    level:         surface
    long_name:     Total Precipitation
    short_name:    APCP_surface
    units:         kg/m^2

In [5]:
# Subset to only include the growing season
subset_2020 = da_yrs_var.sel(time=slice("2020-05-01", "2020-09-26"))
subset_2021 = da_yrs_var.sel(time=slice("2021-04-10", "2021-10-06"))
subset_2022 = da_yrs_var.sel(time=slice("2022-04-15", "2022-10-05"))
subset_2023 = da_yrs_var.sel(time=slice("2023-04-01", "2023-10-18"))
subset_2024 = da_yrs_var.sel(time=slice("2024-04-25", "2024-10-10"))

da_yrs_var_GS_subset = xr.concat([subset_2020, subset_2021, subset_2022, subset_2023, subset_2024], dim="time")

In [6]:
### Clip data to study area ###
gdf = gdf.to_crs(da_yrs_var_GS_subset.rio.crs)  # Reproject if needed
da_yrs_var_GS_subset.rio.write_crs(gdf.crs, inplace=True)  # Set CRS on DataArray

da_yrs_var_GS_subset_clipped = da_yrs_var_GS_subset.rio.clip(
    geometries=gdf.geometry.apply(mapping),
    crs=gdf.crs
)

In [64]:
da_yrs_var_GS_subset_clipped.resample(time="1D").sum()

for t in da_yrs_var_GS_subset_clipped.resample(time="1D").sum().time.values:
    slice_t = da_yrs_var_GS_subset_clipped.resample(time="1D").sum().sel(time=t)
    if (slice_t != 0).any():
        selected_time = t
        break

print(f"Selected time with non-zero values: {selected_time}")

Selected time with non-zero values: 2020-05-01T00:00:00.000000000


In [69]:
da_slice = da_yrs_var_GS_subset_clipped.resample(time="1D").sum().sel(time=selected_time).compute()

df = da_slice.to_dataframe(name="value").reset_index()

test_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs=da_slice.rio.crs  # use .rio.crs if using rioxarray
)

test_gdf.to_file("nonzero_snapshot.shp")

C:\Users\adadkhah\AppData\Local\Temp\ipykernel_16200\1726572878.py:11: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  test_gdf.to_file("nonzero_snapshot.shp")
c:\Users\adadkhah\AppData\Local\miniconda3\envs\AORC_download_2\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field time created as String field, though DateTime requested.
  ogr_write(
c:\Users\adadkhah\AppData\Local\miniconda3\envs\AORC_download_2\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'spatial_ref' to 'spatial_re'
  ogr_write(


In [7]:
# Convert to daily, convert to df, rename the value col, to csv
da_yrs_var_GS_subset_clipped.resample(time="1D").sum()\
    .to_dataframe(name="value").reset_index()\
    .rename(columns={"Value": "AORC_Precipitation"})\
        .to_csv("Daily_AORC_Precipitation_2020-2024.csv", index=False)

c:\Users\adadkhah\AppData\Local\miniconda3\envs\AORC_download_2\Lib\site-packages\dask\array\reductions.py:291: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(x_chunk, axis=axis, keepdims=keepdims)


#### Calculate GDD daily based on daily AORC TA min and max

In [8]:
# Start date - In Year-Month-Day format the earliest start date can be '1979-02-01'
start_datetime = '2020-01-01'

# End date - In Year-Month-Day format the latest end date can be '2023-01-31'
end_datetime = '2024-12-31'

# File path to the shapefile
shapefile_path = "2020_fishnet_raster_extent.shp"

# Variable name to retrieve data (look at the following table for valid variable names)
variable_name = 'TMP_2maboveground'

# User-defined aggregation interval - valid values are 'hour','day','month','year'
agg_interval = 'hour'

# Read the shapefile
gdf = gpd.read_file(shapefile_path)

## Create a list of years to retrieve data 
start_yr = datetime.strptime(start_datetime, '%Y-%m-%d').year
end_yr = datetime.strptime(end_datetime, '%Y-%m-%d').year
yrs = list(range(start_yr, end_yr+1))

## Loading data (AORC data are organized by years, look at https://noaa-nws-aorc-v1-1-1km.s3.amazonaws.com/index.html)
# Base URL
base_url = f's3://noaa-nws-aorc-v1-1-1km'
# Creating a connection to Amazon S3 bucket using the s3fs library (https://s3fs.readthedocs.io/en/latest/api.html).
s3_out = s3fs.S3FileSystem(anon=True)              # access S3 as if it were a file system. 
fileset = [s3fs.S3Map(                             # maps each year's Zarr dataset from S3 to a local-like object.
            root=f"s3://{base_url}/{yr}.zarr",     # Zarr dataset for each year
            s3=s3_out,                             # connection
            check=False                            # checking if the dataset exists before trying to load it
        ) for yr in yrs]                           # loops through each year

## Load data for specified years and veriable of interest using the xarray library
ds_yrs = xr.open_mfdataset(fileset, engine='zarr')
da_yrs_var = ds_yrs[variable_name]
variable_long_name = da_yrs_var.attrs.get('long_name')

In [9]:
### Subset to only include the growing season  ###
subset_2020 = da_yrs_var.sel(time=slice("2020-05-01", "2020-09-26"))
subset_2021 = da_yrs_var.sel(time=slice("2021-04-10", "2021-10-06"))
subset_2022 = da_yrs_var.sel(time=slice("2022-04-15", "2022-10-05"))
subset_2023 = da_yrs_var.sel(time=slice("2023-04-01", "2023-10-18"))
subset_2024 = da_yrs_var.sel(time=slice("2024-04-25", "2024-10-10"))

da_yrs_var_GS_subset = xr.concat([subset_2020, subset_2021, subset_2022, subset_2023, subset_2024], dim="time")

### Clip data to study area ###
gdf = gdf.to_crs(da_yrs_var_GS_subset.rio.crs)  # Reproject if needed
da_yrs_var_GS_subset.rio.write_crs(gdf.crs, inplace=True)  # Set CRS on DataArray

da_yrs_var_GS_subset_clipped = da_yrs_var_GS_subset.rio.clip(
    geometries=gdf.geometry.apply(mapping),
    crs=gdf.crs
)

In [11]:
### Calcuate daily GDD based on Ta min and Ta max ###

tmin = da_yrs_var_GS_subset_clipped.resample(time="1D").min()
tmax = da_yrs_var_GS_subset_clipped.resample(time="1D").max()

ds_daily = xr.Dataset({
    "T_min": tmin,
    "T_max": tmax
})

df = ds_daily.to_dataframe().reset_index()
df["GDD"] = (((df["T_max"] + df["T_min"]-273.15-273.15)) / 2) - 10
df["GDD"] = df["GDD"].clip(lower=0)

In [12]:
### Export the file as csv)
df.to_csv("Daily_AORC_GDD_2020-2024.csv")

In [54]:
df = ds_daily.to_dataframe().reset_index()
df["GDD"] = (((df["T_max"] + df["T_min"]-273.15-273.15)) / 2) - 10
df["GDD"] = df["GDD"].clip(lower=0)
df

,latitude,longitude,time,spatial_ref,T_min,T_max,GDD
0,40.524179,-78.160407,2020-05-01,0,281.700004,288.800004,2.100004
1,40.524179,-78.160407,2020-05-02,0,280.000004,294.700004,4.200004
2,40.524179,-78.160407,2020-05-03,0,287.400004,297.600004,9.350004
3,40.524179,-78.160407,2020-05-04,0,286.100004,295.500004,7.650004
4,40.524179,-78.160407,2020-05-05,0,277.500004,286.900004,0.000000
...,...,...,...,...,...,...,...
844417,40.757503,-77.977081,2023-10-14,0,280.000004,282.200004,0.000000
844418,40.757503,-77.977081,2023-10-15,0,280.000004,284.600004,0.000000
844419,40.757503,-77.977081,2023-10-16,0,279.700004,286.600004,0.000004
844420,40.757503,-77.977081,2023-10-17,0,281.000004,284.700004,0.000000
